In [10]:
import pandas as pd
from pathlib import Path

# Project folders
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "cleaned_data"

# If this notebook is inside a subfolder, use the project root instead
if not DATA_DIR.exists():
    BASE_DIR = BASE_DIR.parent
    DATA_DIR = BASE_DIR / "cleaned_data"

# Dataset paths
sy_path = DATA_DIR / "SY_C_Sem2_2024_25_Attendance.csv"
ty_path = DATA_DIR / "TY_C_Sem2_2025_26_Attendance.csv"

# Load datasets
sy_df = pd.read_csv(sy_path)
ty_df = pd.read_csv(ty_path)

print("SY dataset shape:", sy_df.shape)
print("TY dataset shape:", ty_df.shape)

print("\nSY attendance columns:")
print([c for c in sy_df.columns if "_total_attendance_percent_" in c])

print("\nTY attendance columns:")
print([c for c in ty_df.columns if "_total_attendance_percent_" in c])

SY dataset shape: (83, 41)
TY dataset shape: (170, 40)

SY attendance columns:
['academic_year_2024_25_department_computer_science_and_engineering_start_date_01_july_2024_end_date_31_may_2025_sem_ii_sy_c_total_attendance_percent_unnamed_39_level_5_unnamed_39_level_6']

TY attendance columns:
['academic_year_2025_26_department_computer_science_and_engineering_start_date_01_july_2025_end_date_31_may_2026_sem_ii_ty_c_total_attendance_percent_unnamed_39_level_5_unnamed_39_level_6']


In [11]:
# Find the required columns in each dataset
def get_attendance_data(df):
    name_col = next(
        c for c in df.columns if "_student_name_" in c
    )
    roll_col = next(
        c for c in df.columns if "_roll_no_" in c
    )
    attendance_col = next(
        c for c in df.columns
        if "_total_attendance_percent_" in c
    )

    # Keep only the required columns
    data = df[[name_col, roll_col, attendance_col]].copy()

    data.columns = ["Student_Name", "Roll_Number", "Overall_Attendance"]

    # Clean student names
    data["Student_Name"] = (
        data["Student_Name"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Convert attendance into numeric values
    data["Overall_Attendance"] = pd.to_numeric(
        data["Overall_Attendance"]
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    # Remove missing or invalid attendance values
    data = data.dropna(
        subset=["Student_Name", "Overall_Attendance"]
    )

    data = data[
        data["Overall_Attendance"].between(0, 100)
        & data["Student_Name"].ne("")
    ]

    return data


# Prepare SY and TY data
sy_attendance = get_attendance_data(sy_df)
ty_attendance = get_attendance_data(ty_df)

print("SY records:", len(sy_attendance))
print("TY records:", len(ty_attendance))

print("\nSY attendance preview:")
display(sy_attendance.head())

print("\nTY attendance preview:")
display(ty_attendance.head())

SY records: 83
TY records: 83

SY attendance preview:


,Student_Name,Roll_Number,Overall_Attendance
0,TEJAS ALAWANI,SYC01,66.52
1,NIRJALA PUJARI,SYC02,86.88
2,SAHIL CHILE,SYC03,78.73
3,SANIKA RANAGE,SYC04,82.35
4,SAMIKSHA PAVASKAR,SYC05,73.76



TY attendance preview:


,Student_Name,Roll_Number,Overall_Attendance
0,TEJAS ALAWANI,TYC01,77.24
1,NIRJALA PUJARI,TYC02,88.99
2,SAHIL CHILE,TYC03,86.9
3,SANIKA RANAGE,TYC04,83.49
4,SAMIKSHA PAVASKAR,TYC05,82.57


In [12]:
# Save cleaned datasets as separate CSV files
sy_attendance.to_csv(
    DATA_DIR / "SY_Overall_Attendance_Cleaned.csv",
    index=False
)

ty_attendance.to_csv(
    DATA_DIR / "TY_Overall_Attendance_Cleaned.csv",
    index=False
)

print("Cleaned datasets saved successfully!")

Cleaned datasets saved successfully!


In [13]:
# Check for duplicate student names before matching
print("Duplicate names in SY:")
display(
    sy_attendance[
        sy_attendance["Student_Name"].duplicated(keep=False)
    ].sort_values("Student_Name")
)

print("Duplicate names in TY:")
display(
    ty_attendance[
        ty_attendance["Student_Name"].duplicated(keep=False)
    ].sort_values("Student_Name")
)

# Keep only names that appear once in each dataset
sy_unique = sy_attendance.drop_duplicates(
    subset="Student_Name", keep=False
).copy()

ty_unique = ty_attendance.drop_duplicates(
    subset="Student_Name", keep=False
).copy()

# Match students by cleaned name
matched_data = pd.merge(
    sy_unique[["Student_Name", "Overall_Attendance"]],
    ty_unique[["Student_Name", "Overall_Attendance"]],
    on="Student_Name",
    how="inner",
    suffixes=("_SY", "_TY"),
    validate="one_to_one"
)

print("\nSY unique students:", len(sy_unique))
print("TY unique students:", len(ty_unique))
print("Matched students:", len(matched_data))

display(matched_data.head(10))

Duplicate names in SY:


,Student_Name,Roll_Number,Overall_Attendance
18,YASH PATIL,SYC19,68.33
46,YASH PATIL,SYC48,72.4


Duplicate names in TY:


,Student_Name,Roll_Number,Overall_Attendance
18,YASH PATIL,TYC19,73.1
46,YASH PATIL,TYC48,75.17



SY unique students: 81
TY unique students: 81
Matched students: 81


,Student_Name,Overall_Attendance_SY,Overall_Attendance_TY
0,TEJAS ALAWANI,66.52,77.24
1,NIRJALA PUJARI,86.88,88.99
2,SAHIL CHILE,78.73,86.9
3,SANIKA RANAGE,82.35,83.49
4,SAMIKSHA PAVASKAR,73.76,82.57
5,MAITHILI DHOPATE,82.35,84.4
6,AMRUTA POWAR,88.24,85.32
7,DHIREN ADE,55.66,73.39
8,PRANAV NIMBALKAR,76.47,77.93
9,SATYAJEET GONUGADE,81.45,77.24


In [14]:
# Check dataset size and column names
print("Matched dataset shape:", matched_data.shape)
print("\nColumns:", matched_data.columns.tolist())

# Check missing values
print("\nMissing values:")
print(matched_data.isnull().sum())

# Check attendance statistics
print("\nAttendance statistics:")
display(
    matched_data[
        ["Overall_Attendance_SY", "Overall_Attendance_TY"]
    ].describe().round(2)
)

Matched dataset shape: (81, 3)

Columns: ['Student_Name', 'Overall_Attendance_SY', 'Overall_Attendance_TY']

Missing values:
Student_Name             0
Overall_Attendance_SY    0
Overall_Attendance_TY    0
dtype: int64

Attendance statistics:


,Overall_Attendance_SY,Overall_Attendance_TY
count,81.0,81.0
mean,76.96,78.39
std,8.76,8.32
min,55.66,59.63
25%,73.3,73.39
50%,78.73,78.9
75%,82.35,83.45
max,94.12,98.17


In [15]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Input feature: previous year's overall attendance
X = matched_data[["Overall_Attendance_SY"]]

# Target: next year's overall attendance
y = matched_data["Overall_Attendance_TY"]

# Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=3,
    random_state=42
)

# XGBoost model
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=5,
    objective="reg:squarederror",
    random_state=42,
    verbosity=0
)

# Five-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rf_predictions = cross_val_predict(rf_model, X, y, cv=cv)
xgb_predictions = cross_val_predict(xgb_model, X, y, cv=cv)

# Calculate model weights using validation MAE
rf_mae = mean_absolute_error(y, rf_predictions)
xgb_mae = mean_absolute_error(y, xgb_predictions)

rf_weight = (1 / rf_mae) / ((1 / rf_mae) + (1 / xgb_mae))
xgb_weight = 1 - rf_weight

# Hybrid predictions
hybrid_predictions = (
    rf_weight * rf_predictions
    + xgb_weight * xgb_predictions
)

# Evaluation function
def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    return {
        "Model": name,
        "MAE": round(mae, 3),
        "RMSE": round(rmse, 3),
        "R2": round(r2, 3)
    }

results = pd.DataFrame([
    evaluate_model("Random Forest", y, rf_predictions),
    evaluate_model("XGBoost", y, xgb_predictions),
    evaluate_model("Hybrid", y, hybrid_predictions)
])

print(f"Matched students: {len(matched_data)}")
print(f"Random Forest weight: {rf_weight:.3f}")
print(f"XGBoost weight: {xgb_weight:.3f}")

display(results)

Matched students: 81
Random Forest weight: 0.500
XGBoost weight: 0.500


,Model,MAE,RMSE,R2
0,Random Forest,3.752,4.736,0.672
1,XGBoost,3.759,4.780,0.665
2,Hybrid,3.667,4.655,0.683


In [16]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Create the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=3,
    random_state=42
)

# Create the XGBoost model
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=5,
    objective="reg:squarederror",
    random_state=42,
    verbosity=0
)

print("Both models created successfully!")

Both models created successfully!


In [17]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("RF weight:", rf_weight)
print("XGBoost weight:", xgb_weight)

X shape: (81, 1)
y shape: (81,)
RF weight: 0.5004296192550134
XGBoost weight: 0.4995703807449866


In [18]:
import joblib
from pathlib import Path

# Define the base directory as the current notebook's working directory
BASE_DIR = Path.cwd()

# Create a separate folder for this model
MODEL_DIR = BASE_DIR / "saved_models_overall"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Train final models using all matched historical data
rf_model.fit(X, y)
xgb_model.fit(X, y)

# Save models and hybrid weights
joblib.dump(rf_model, MODEL_DIR / "overall_rf.pkl")
joblib.dump(xgb_model, MODEL_DIR / "overall_xgb.pkl")

joblib.dump(
    {"rf_weight": rf_weight, "xgb_weight": xgb_weight},
    MODEL_DIR / "overall_hybrid_weights.pkl"
)

# Save the expected feature name
joblib.dump(
    ["Overall_Attendance_SY"],
    MODEL_DIR / "overall_hybrid_features.pkl"
)

print("Models saved successfully!")
print("Saved location:", MODEL_DIR.resolve())
print("\nFiles:")
for file in MODEL_DIR.iterdir():
    print(file.name)

Models saved successfully!
Saved location: C:\Users\Tanvi\OneDrive\Desktop\Megaproject\saved_models_overall

Files:
overall_hybrid_features.pkl
overall_hybrid_weights.pkl
overall_rf.pkl
overall_xgb.pkl


In [19]:
def predict_next_year_attendance(input_df):
    """
    Predict next-year overall attendance from a CSV-style DataFrame.

    Required columns:
    Student_Name, Roll_Number, Overall_Attendance
    """

    required_columns = [
        "Student_Name",
        "Roll_Number",
        "Overall_Attendance"
    ]

    missing = [
        col for col in required_columns
        if col not in input_df.columns
    ]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    attendance = pd.to_numeric(
        input_df["Overall_Attendance"],
        errors="coerce"
    )

    if attendance.isna().any() or not attendance.between(0, 100).all():
        raise ValueError(
            "Overall_Attendance must contain valid percentages from 0 to 100."
        )

    # Prepare the model input
    X_new = pd.DataFrame({
        "Overall_Attendance_SY": attendance
    })

    # Predict with both trained models
    rf_pred = rf_model.predict(X_new)
    xgb_pred = xgb_model.predict(X_new)

    # Combine predictions using the saved hybrid weights
    predictions = (
        rf_weight * rf_pred
        + xgb_weight * xgb_pred
    )

    # Keep results within the valid attendance range
    predictions = np.clip(predictions, 0, 100)

    # Return student identifiers and predictions
    result = input_df[["Student_Name", "Roll_Number"]].copy()
    result["Predicted_Next_Year_Attendance"] = np.round(predictions, 2)

    return result

In [20]:
sample_data = pd.DataFrame({
    "Student_Name": ["Student A", "Student B", "Student C"],
    "Roll_Number": ["TEST01", "TEST02", "TEST03"],
    "Overall_Attendance": [65.0, 78.5, 90.0]
})

sample_predictions = predict_next_year_attendance(sample_data)

display(sample_predictions)

,Student_Name,Roll_Number,Predicted_Next_Year_Attendance
0,Student A,TEST01,67.74
1,Student B,TEST02,79.55
2,Student C,TEST03,88.91


In [24]:
import importlib
import attendance_predictor

importlib.reload(attendance_predictor)

print(attendance_predictor.__file__)

c:\Users\Tanvi\OneDrive\Desktop\Megaproject\attendance_predictor.py


In [25]:
import pandas as pd

test_data = pd.DataFrame({
    "Student_Name": ["Student A", "Student B", "Student C"],
    "Roll_Number": ["TEST01", "TEST02", "TEST03"],
    "Overall_Attendance": [65.0, 78.5, 90.0]
})

results = attendance_predictor.predict_attendance(test_data)
display(results)

,Student_Name,Roll_Number,Predicted_Next_Year_Attendance
0,Student A,TEST01,67.74
1,Student B,TEST02,79.55
2,Student C,TEST03,88.91


In [27]:
from attendance_predictor import predict_csv

results = predict_csv(
    input_path="sample_attendance.csv",
    output_path="predicted_attendance.csv"
)

display(results)

print("Predictions saved to predicted_attendance.csv")

Predictions saved to: predicted_attendance.csv


,Student_Name,Roll_Number,Predicted_Next_Year_Attendance
0,Student A,TEST01,67.74
1,Student B,TEST02,79.55
2,Student C,TEST03,88.91


Predictions saved to predicted_attendance.csv
